# 02 — Data quality and cleaning

**Objective.** Make source-specific checks and transformations visible.

**Business relevance.** Establish a transparent descriptive evidence base without merging incompatible populations or implying causality.

**Data source.** Published processed tables plus redistributable raw files  
**Period.** Source dependent  
**Granularity.** Table-specific keys  
**Units.** Persons, percent, million EUR and EUR per resident

Last updated: 2026-08-25

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed'
plt.style.use('seaborn-v0_8-whitegrid')

In [2]:
tables={p.stem:pd.read_csv(p) for p in DATA.glob('*.csv')}
quality=[]
for name,df in tables.items():
    quality.append({'table':name,'rows':len(df),'columns':len(df.columns),'missing_cells':int(df.isna().sum().sum()),'duplicate_rows':int(df.duplicated().sum())})
display(pd.DataFrame(quality))

,table,rows,columns,missing_cells,duplicate_rows
0,ema_glp1_eu_authorisations,6,9,0,0
1,fact_disease_cost_observed,8,14,0,0
2,fact_obesity_observed,9,15,0,0
3,fact_population_observed,435,12,0,0
4,fact_population_state_age_sex,14560,15,0,0


## Cleaning decisions

German decimal separators are converted only for source columns that require it. Composite periods remain categorical. Missing values are never converted to zero. Keys, units and expected periods are validated before analysis.

In [3]:
obesity=tables['fact_obesity_observed'].copy()
assert obesity['date_key'].is_unique
assert obesity['unit'].eq('percent').all()
assert (obesity.ci95_lower_pct <= obesity.estimate_pct).all() and (obesity.estimate_pct <= obesity.ci95_upper_pct).all()
costs=tables['fact_disease_cost_observed'].copy()
assert not costs.duplicated(['year','diagnosis_code','metric_code']).any()
print('Validated obesity rows:',len(obesity),'Validated cost rows:',len(costs))

Validated obesity rows: 9 Validated cost rows: 8


## Limitations, outputs and conclusion

The analysis preserves source periods, units and denominators; it does not impute, interpolate or claim causality. Outputs are the displayed summary tables and Matplotlib figures. The conclusion is descriptive and should be read with the source-specific limitations above.